In [10]:
!pip install -r requirements.txt -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
)
output = llm.invoke('Explain quantum mechanics in one sentence')
print(output.content)


Quantum mechanics is the branch of physics that describes the behavior of matter and energy at the smallest scales, where particles exhibit wave‑particle duality, occupy discrete energy levels, and evolve according to probabilistic wavefunctions.


In [13]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage(content = "You are a physicist and respond only in German."),
    HumanMessage(content = "Explain quantum mechanics in one sentence")
]
response = llm.invoke(messages)
print(response.content)

Quantenmechanik beschreibt die physikalischen Phänomene auf atomarer und subatomarer Ebene durch Wahrscheinlichkeitsamplituden, die in Wellenfunktionen dargestellt werden und sich nach dem Prinzip der Superposition sowie der Unschärferelation verhalten.


## In Memory Cache

In [14]:
from langchain_core.globals import set_llm_cache
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
)

In [20]:
%%time
from langchain_community.cache import InMemoryCache
set_llm_cache(InMemoryCache())
prompt = 'Tell me a joke that would make a computer science student laugh'
output = llm.invoke(prompt)

CPU times: user 6.43 ms, sys: 9.16 ms, total: 15.6 ms
Wall time: 653 ms


In [22]:
%%time
output = llm.invoke(prompt)

CPU times: user 336 μs, sys: 1.27 ms, total: 1.61 ms
Wall time: 1.61 ms


## SQLite Caching

In [24]:
from langchain_community.cache import SQLiteCache
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

# First req not in cache will take time
llm.invoke("Tell me a joke")

# Second req in cache so faster
llm.invoke("Tell me a joke")


AIMessage(content='Why don’t skeletons fight each other?  \n\nThey don’t have the guts!', additional_kwargs={'reasoning_content': 'User: "Tell me a joke". We just need to produce a joke. The policy: no disallowed content. It\'s safe. Just produce a joke. We\'ll comply.'}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 75, 'total_tokens': 137, 'completion_time': 0.067172039, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.003511808, 'prompt_tokens_details': None, 'queue_time': 0.043190052, 'total_time': 0.070683847}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0bc5-8b28-7fd0-abed-e91a8dd75015-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 62, 'total_tokens': 137, 'output_token_details': {'reasoning': 36}, 'total_cost': 0})

## LLM Streaming

In [25]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
)
prompt = "Write a song about moon"
print(llm.invoke(prompt).content)

**Title: “Silver Lullaby”**

*(Verse 1)*  
When the city lights flicker and fade,  
I hear the quiet hum of a distant shade.  
High above the rooftops, where the night is born,  
A silver eye watches over us all.

*(Pre‑Chorus)*  
It’s a quiet witness to every dream,  
A silent promise that the darkness gleams.  
With every heartbeat, it pulls the tide,  
Guiding us gently on the other side.

*(Chorus)*  
Oh moon, you’re the silver lullaby,  
Washing over the world in a gentle sigh.  
You rise, you fall, you never lie—  
In the dark, you’re the light we can’t deny.

*(Verse 2)*  
I’ve walked the streets in your glow so pale,  
Seeing shadows dance on the window rail.  
Your craters hold stories of forgotten seas,  
Of lovers who whispered in your quiet breeze.

*(Pre‑Chorus)*  
You’re the compass in a stormy night,  
The keeper of secrets that feel so right.  
When the world feels heavy, you’re a soft refrain,  
A promise that the sun will rise again.

*(Chorus)*  
Oh moon, you’re the 

In [26]:
for chunk in llm.stream(prompt):
    print(chunk.content, end="", flush=True)

**Title: “Silver Lullaby”**

*(Verse 1)*  
In the hush of twilight, when the city’s glow is dim,  
A quiet silver river starts to rise from within.  
She drifts above the rooftops, a lantern in the night,  
Guiding wandering dreamers with her gentle, pale light.  

*(Pre‑Chorus)*  
She watches over lovers, over secrets that we keep,  
Whispering to the shadows, “You’re not alone in sleep.”  

*(Chorus)*  
Oh, Moon, you’re the song that the stars have sung,  
A silver lullaby that rides the night’s soft tongue.  
You pull the tide, you paint the night sky blue,  
In every quiet moment, I feel the pull of you.  

*(Verse 2)*  
She’s the silent witness to the promises we make,  
The silver mirror of a heart that’s wide awake.  
When the world feels heavy, when the day is too long,  
She hums a lullaby that turns the dark to song.  

*(Pre‑Chorus)*  
She paints the rooftops gold, she turns the night to blue,  
A quiet lull‑song that whispers, “I’m with you.”  

*(Chorus)*  
Oh, Moon, you’r

## Prompt Templates

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
template = '''You are an experienced virologist.
Write a few sentences about the following virus {virus} in {language}.'''

prompt_template = PromptTemplate.from_template(template = template)

prompt = prompt_template.format(virus = "Covid 19", language = "German")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
)

for chunk in llm.stream(prompt):
    print(chunk.content, end="", flush=True)


COVID‑19 ist die durch das neuartige Coronavirus SARS‑CoV‑2 verursachte Atemwegserkrankung. Das Virus gehört zur Gattung Coronaviridae und hat sich erstmals im Dezember 2019 in Wuhan, China, verbreitet. Es überträgt sich hauptsächlich durch Tröpfchen und Aerosole, kann aber auch an Oberflächen überleben. Die klinische Präsentation reicht von asymptomatischen Fällen bis zu schwerem Pneumonien mit Multi‑Organ‑Schäden. Durch weltweite Impfkampagnen, antivirale Therapien und Hygiene‑Maßnahmen konnte die Mortalität deutlich reduziert werden, jedoch bleibt COVID‑19 ein bedeutendes globales Gesundheitsproblem.
COVID‑19 ist die durch das neuartige Coronavirus SARS‑CoV‑2 verursachte Infektionskrankheit.  
SARS‑CoV‑2 gehört zur Familie der Coronaviridae, besitzt ein einzelsträngiges, positives RNA‑Genom und eine lipidische Hülle, die mit Spike‑Proteinen ausgestattet ist, die die Zellaufnahme über den ACE2‑Rezeptor ermöglichen.  
Die Pathogenese ist geprägt von einer starken Immunantwort, die bei

In [30]:
prompt2 = prompt_template.format(virus = "HIV", language = "English")
for chunk in llm.stream(prompt2):
    print(chunk.content, end="", flush=True)

Human Immunodeficiency Virus (HIV) is a lentivirus of the Retroviridae family that infects human immune cells, primarily CD4⁺ T helper lymphocytes, macrophages, and dendritic cells. The virus enters cells via the CD4 receptor and a coreceptor (CCR5 or CXCR4), integrates its RNA genome into the host DNA, and hijacks cellular machinery to produce new virions. Chronic infection leads to progressive depletion of CD4⁺ cells, impairing immune surveillance and culminating in Acquired Immunodeficiency Syndrome (AIDS). While antiretroviral therapy (ART) can suppress viral replication and restore immune function, no cure currently exists, making prevention and early treatment essential.

## Chat Prompt Templates

In [35]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You respond only in JSON format."),
        HumanMessagePromptTemplate.from_template('Top {n} countries in {area} by population.'),
    ]
)
messages = chat_template.format_messages(n='10', area='Europe')
print(messages)

[SystemMessage(content='You respond only in JSON format.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Top 10 countries in Europe by population.', additional_kwargs={}, response_metadata={})]


In [36]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
)
output = llm.invoke(messages)
print(output.content)

```json
{
  "top_10_european_countries_by_population": [
    { "country": "Russia", "population_millions": 146.0 },
    { "country": "Germany", "population_millions": 84.0 },
    { "country": "Turkey", "population_millions": 82.0 },
    { "country": "United Kingdom", "population_millions": 67.0 },
    { "country": "France", "population_millions": 65.0 },
    { "country": "Italy", "population_millions": 60.0 },
    { "country": "Spain", "population_millions": 47.0 },
    { "country": "Ukraine", "population_millions": 41.0 },
    { "country": "Poland", "population_millions": 38.0 },
    { "country": "Romania", "population_millions": 19.0 }
  ]
}
```


## SIMPLE CHAINS

In [38]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
)

template = '''You are an experienced virologist.
Write a few sentences about the following virus {virus} in {language}.'''

prompt = PromptTemplate.from_template(template = template)

chain = prompt | llm
output = chain.invoke({'virus': 'Covid-19','language': 'English'})

In [39]:
print(output.content)

COVID‑19 is caused by the SARS‑CoV‑2 virus, a betacoronavirus with a single‑stranded positive‑sense RNA genome of approximately 30 kb. Its spike protein mediates entry into host cells via the ACE2 receptor, and the virus has a high propensity for mutation and recombination, which has driven the emergence of multiple variants of concern. Clinically, SARS‑CoV‑2 can cause a spectrum from asymptomatic infection to severe acute respiratory distress syndrome, with significant morbidity and mortality worldwide, especially among the elderly and immunocompromised.
